In [1]:
#Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
import os

dataset_dir = '/kaggle/input/widstrain'
files = os.listdir(dataset_dir)
print(files)

['train.csv']


In [3]:
# Loading the CSV file
file_path = '/kaggle/input/widstrain/train.csv'  
df = pd.read_csv(file_path)

In [4]:
df.head()

,patient_id,patient_race,payer_type,patient_state,patient_zip3,patient_age,patient_gender,bmi,breast_cancer_diagnosis_code,breast_cancer_diagnosis_desc,...,race_other,race_multiple,hispanic,disabled,poverty,limited_english,commute_time,health_uninsured,veteran,treatment_pd
0,994155,Asian,COMMERCIAL,CA,917,46,F,27.0,C50811,Malignant neoplasm of ovrlp sites of right fem...,...,18.858696,11.426087,47.726087,9.895652,10.515217,12.745652,32.530435,7.263043,3.810870,35
1,154389,NaN,MEDICARE ADVANTAGE,OH,451,63,F,NaN,C50412,Malig neoplasm of upper-outer quadrant of left...,...,0.255319,2.234043,1.182979,18.317021,13.546809,0.146809,31.890909,7.631915,9.631915,33
2,387343,NaN,COMMERCIAL,TX,773,53,F,NaN,C50212,Malig neoplasm of upper-inner quadrant of left...,...,3.588679,7.915094,21.064151,14.083019,11.943396,2.549057,32.556250,16.396226,10.392453,24
3,921275,Hispanic,MEDICAID,CA,928,50,F,NaN,1749,"Malignant neoplasm of breast (female), unspeci...",...,11.645455,10.081818,37.948485,8.957576,10.109091,8.057576,30.606061,7.018182,4.103030,455
4,803454,NaN,COMMERCIAL,NY,112,39,F,18.0,1749,"Malignant neoplasm of breast (female), unspeci...",...,9.184211,6.089474,18.960526,10.194737,18.642105,14.173684,42.502632,6.392105,1.755263,162


In [5]:
desired_columns=['metastatic_first_treatment', 'breast_cancer_diagnosis_code', 'breast_cancer_diagnosis_year',
                   'breast_cancer_diagnosis_desc','metastatic_cancer_diagnosis_code','treatment_pd']

In [6]:
df=df[desired_columns]

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Initializing CountVectorizer
count_vectorizer = CountVectorizer()

# Fitting and transforming the text data
diagnosis_desc_count = count_vectorizer.fit_transform(df['breast_cancer_diagnosis_desc'])

# Converting the sparse matrix to a dense matrix
diagnosis_desc_count_dense = diagnosis_desc_count.toarray()

# Standardizing the data
scaler = StandardScaler()
diagnosis_desc_count_standardized = scaler.fit_transform(diagnosis_desc_count_dense)

# Applying PCA
pca = PCA(n_components=4)  
diagnosis_desc_pca = pca.fit_transform(diagnosis_desc_count_standardized)

# Converting the PCA result to a DataFrame 
diagnosis_desc_pca_df = pd.DataFrame(diagnosis_desc_pca, columns=['PC1', 'PC2','PC3','PC4'])

# Dropping the original text column
df.drop(columns=['breast_cancer_diagnosis_desc'], inplace=True)

# Adding the PCA columns to the original DataFrame
df = pd.concat([df, diagnosis_desc_pca_df], axis=1)

In [8]:
# Separating predictors (X) and target (y)
X = df.drop(columns=['treatment_pd'])  # X contains all columns except 'treatment_pd'
y = df['treatment_pd']  # y contains only the 'treatment_pd' column

In [9]:
X.head()

,metastatic_first_treatment,breast_cancer_diagnosis_code,breast_cancer_diagnosis_year,metastatic_cancer_diagnosis_code,PC1,PC2,PC3,PC4
0,DOXORUBICIN HCL,C50811,2018,C779,-1.322668,-0.485786,3.110523,-1.617703
1,DOXORUBICIN HCL,C50412,2018,C7951,2.777515,0.098504,0.156421,2.196646
2,PACLITAXEL,C50212,2018,C773,2.613165,0.077032,0.194500,2.163953
3,GEMCITABINE HCL,1749,2015,C787,-1.833837,0.637013,-0.513029,0.803617
4,DOXORUBICIN HCL,1749,2015,C7989,-1.833837,0.637013,-0.513029,0.803617


In [10]:
categorical_cols = ['metastatic_first_treatment','breast_cancer_diagnosis_code', 'metastatic_cancer_diagnosis_code']
for col in categorical_cols:
    X[col] = X[col].fillna('missing')  # Replacing NaN with a placeholder value

In [11]:
# Splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# Training CatBoost regression model with CatBoost encoding
train_pool = Pool(data=X_train, label=y_train, cat_features=categorical_cols)
test_pool = Pool(data=X_test, label=y_test, cat_features=categorical_cols)

In [13]:
from catboost import CatBoostRegressor

model1 = CatBoostRegressor(cat_features=categorical_cols,
                           verbose=100)

# Fitting the model
model1.fit(train_pool, eval_set=test_pool)

# Predicting on the test set
y_pred = model1.predict(X_test)

# Calculating RMSE
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"RMSE on test set: {rmse:.4f}")

Learning rate set to 0.082811
0:	learn: 183.1005150	test: 178.1638463	best: 178.1638463 (0)	total: 72.8ms	remaining: 1m 12s
100:	learn: 141.7162787	test: 138.6217843	best: 138.6217843 (100)	total: 1.14s	remaining: 10.1s
200:	learn: 140.0251675	test: 138.4623207	best: 138.4544772 (192)	total: 2.01s	remaining: 8s
300:	learn: 138.6534457	test: 138.7522353	best: 138.4544772 (192)	total: 2.9s	remaining: 6.74s
400:	learn: 137.3494826	test: 138.8586950	best: 138.4544772 (192)	total: 3.79s	remaining: 5.66s
500:	learn: 136.1235561	test: 138.9948880	best: 138.4544772 (192)	total: 4.68s	remaining: 4.66s
600:	learn: 135.2572608	test: 139.2051956	best: 138.4544772 (192)	total: 5.59s	remaining: 3.71s
700:	learn: 134.3608123	test: 139.3834926	best: 138.4544772 (192)	total: 6.5s	remaining: 2.77s
800:	learn: 133.4512955	test: 139.5307541	best: 138.4544772 (192)	total: 7.39s	remaining: 1.84s
900:	learn: 132.6657147	test: 139.6712386	best: 138.4544772 (192)	total: 8.3s	remaining: 913ms
999:	learn: 131.84